# 01 — Bronze Layer: Raw Data Ingestion

**Purpose:** Ingest raw sales data from source files into the Lakehouse Bronze layer.  
**Output:** Raw files stored in `Files/bronze/` (no transformation applied).  
**Schedule:** Run daily at 01:00 UTC via Fabric Pipeline.


In [ ]:
# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────
LAKEHOUSE_NAME = "sales_lakehouse"
BRONZE_PATH    = f"Files/bronze"
SOURCE_PATH    = f"Files/raw"   # Drop zone for incoming files

from datetime import datetime
RUN_DATE = datetime.utcnow().strftime('%Y%m%d')
print(f"Run date: {RUN_DATE}")

In [ ]:
# ─────────────────────────────────────────────
# 1. Read raw Sales CSV from drop zone
# ─────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

spark = SparkSession.builder.getOrCreate()

# Read sales file
df_sales_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com/{SOURCE_PATH}/sales/*.csv")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("sales"))
    .withColumn("_run_date", lit(RUN_DATE))
)

print(f"Sales rows ingested: {df_sales_raw.count():,}")
df_sales_raw.printSchema()

In [ ]:
# ─────────────────────────────────────────────
# 2. Read raw Customer CSV
# ─────────────────────────────────────────────
df_customers_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com/{SOURCE_PATH}/customers/*.csv")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("customers"))
    .withColumn("_run_date", lit(RUN_DATE))
)

print(f"Customer rows ingested: {df_customers_raw.count():,}")

In [ ]:
# ─────────────────────────────────────────────
# 3. Read raw Products CSV
# ─────────────────────────────────────────────
df_products_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com/{SOURCE_PATH}/products/*.csv")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("products"))
    .withColumn("_run_date", lit(RUN_DATE))
)

print(f"Product rows ingested: {df_products_raw.count():,}")

In [ ]:
# ─────────────────────────────────────────────
# 4. Write to Bronze layer (partitioned by run_date)
#    We write as Delta for ACID compliance even at bronze
# ─────────────────────────────────────────────
def write_bronze(df, name):
    path = f"abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com/{BRONZE_PATH}/{name}"
    (
        df.write
        .format("delta")
        .mode("append")
        .partitionBy("_run_date")
        .save(path)
    )
    print(f"✅ Written bronze/{name} — partition _run_date={RUN_DATE}")

write_bronze(df_sales_raw,     "sales")
write_bronze(df_customers_raw, "customers")
write_bronze(df_products_raw,  "products")

In [ ]:
# ─────────────────────────────────────────────
# 5. Log ingestion summary
# ─────────────────────────────────────────────
summary = [
    {"table": "sales",     "rows": df_sales_raw.count(),     "run_date": RUN_DATE},
    {"table": "customers", "rows": df_customers_raw.count(), "run_date": RUN_DATE},
    {"table": "products",  "rows": df_products_raw.count(),  "run_date": RUN_DATE},
]

df_log = spark.createDataFrame(summary)
(
    df_log.write
    .format("delta")
    .mode("append")
    .save(f"abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com/Files/logs/ingestion_log")
)

df_log.show()
print("\n🎉 Bronze ingestion complete!")